# Search-13 — Groupes automatiques : le problème du mot par automates

**Série Search · Part 1 — Fondations** · Carnet d'accrétion (Pli 4 de l'EPIC Origami #20404)

[← Retour à la série](../README.md) · Précédent : [Search-12a — Composer des regards](Search-12a-Composer-Regards.ipynb) · Suivant : [Search-13a — Traverser un mur](Search-13a-Traverser-Murs-Certifies.ipynb) | [jumeau C#](Search-13-AutomaticGroups-WordProblem-CSharp.ipynb)

Quand la structure du groupe donne l'automate, *décider* d'une propriété d'un mot devient un test
d'appartenance à un langage **régulier** — linéaire en la longueur du mot — au lieu d'une **exploration**
du graphe de Cayley. Ce carnet rend calculable cette frontière, sur les deux groupes types :
`Z^2` (croissance polynômiale) et `F_2` (croissance exponentielle).

> Provenance : Epstein, Cannon, Holt, Levy, Paterson, Thurston, *Word Processing in Groups* (1992) ;
> structure automatique de Gromov. Aucun carnet existant du dépôt ne couvre les groupes automatiques
> (problème du mot par automates, cônes géodésiques, croissance) — vérifié par grep à l'ouverture de #25674.

## Enjeu — la heuristique ultime

Les carnets frères de la série posent le contraste exploration :

- **Search-02** (non informé) explore par **BFS** — il *parcourt* l'espace d'états ;
- **Search-03** (informé) explore par **A\*** — une heuristique *guide* le parcours, mais on explore toujours.

Ce carnet ajoute la couche qui manque : **quand le groupe lui-même fournit l'automate**, il n'y a plus
d'exploration. La structure algébrique *est* la heuristique, et elle est complète : elle transforme la
question « ce mot est-il géodésique ? » en une lecture gauche-droite d'un automate fini, en **O(|mot|)**.


## 1. Le cadre — deux groupes, un alphabet de 4 lettres

On travaille sur l'alphabet `{a, a⁻¹, b, b⁻¹}` (deux générateurs et leurs inverses).

- **`Z^2`** (libre abélien de rang 2) : un mot vaut l'identité si la *somme des exposants* de `a` est 0
  ET celle de `b` est 0. Le groupe est commutatif : `a b a⁻¹ b⁻¹ = 1`.
- **`F_2`** (groupe libre de rang 2) : un mot se réduit par *annulation libre* — on efface chaque lettre
  adjacente à son inverse. `a b a⁻¹` ne se réduit pas (le `b` bloque), mais `a a⁻¹` s'annule.


In [1]:
# Alphabet : deux générateurs et leurs inverses. Un mot = LISTE de tokens.
INV = {'a': 'a-1', 'a-1': 'a', 'b': 'b-1', 'b-1': 'b'}

def word(s):
    """Parse un mot compact en liste de tokens : 'aa-1' -> ['a', 'a-1']."""
    toks, i = [], 0
    while i < len(s):
        if i + 2 < len(s) and s[i + 1] == '-' and s[i + 2] == '1':
            toks.append(s[i] + '-1'); i += 3
        else:
            toks.append(s[i]); i += 1
    return toks

def eval_z2(word):
    """Image d'un mot (liste de tokens) dans Z^2 : (somme exp. a, somme exp. b)."""
    x = y = 0
    for L in word:
        if L == 'a': x += 1
        elif L == 'a-1': x -= 1
        elif L == 'b': y += 1
        elif L == 'b-1': y -= 1
    return (x, y)

def reduce_f2(word):
    """Réduction libre dans F_2 : annule chaque token adjacent à son inverse. Retourne la liste réduite."""
    st = []
    for L in word:
        if st and st[-1] == INV[L]:
            st.pop()
        else:
            st.append(L)
    return st

# Contrôles de base
assert eval_z2(word('aba-1b-1')) == (0, 0)            # a b a^{-1} b^{-1} = 1 dans Z^2 (commutatif)
assert reduce_f2(word('aa-1')) == []                   # annulation libre
assert reduce_f2(word('abb-1')) == word('a')            # b b^{-1} s'annule, reste 'a'
print("Z^2 : a b a^-1 b^-1 vaut", eval_z2(word('aba-1b-1')))
print("F_2 : 'abb-1' se réduit en", reduce_f2(word('abb-1')))


Z^2 : a b a^-1 b^-1 vaut (0, 0)
F_2 : 'abb-1' se réduit en ['a']


### Lecture du résultat

Dans `Z^2` le mot `a b a⁻¹ b⁻¹` vaut l'identité *par commutativité* (la somme des exposants est (0,0)),
alors que dans `F_2` le même enchaînement ne se réduit **pas** — la lettre `b` centrale bloque l'annulation
`a a⁻¹`. C'est toute la différence entre un groupe commutatif et un groupe libre : la *géométrie* des mots
n'y est pas du tout la même, et c'est précisément elle que les automates vont capturer.


## 2. Le problème du mot par automates — l'accepteur **construit**

Le langage des **mots géodésiques** (librement réduits) de `Z^2` est **régulier** : on le reconnaît par un
automate fini, et on le *construit* ici — l'état de l'automate n'est rien d'autre que **la dernière lettre
lue** (plus l'état de départ et un état mort pour les mots qui annulent).


In [2]:
# DFA CONSTRUIT des mots géodésiques (librement réduits) de Z^2.
# Etats : S (depart) + un etat par derniere lettre + D (etat mort, mot qui s'annule).
DELTA = {
    'S':   {'a': 'a',   'a-1': 'a-1', 'b': 'b',   'b-1': 'b-1'},
    'a':   {'a': 'a',   'a-1': 'D',   'b': 'b',   'b-1': 'b-1'},
    'a-1': {'a': 'D',   'a-1': 'a-1', 'b': 'b',   'b-1': 'b-1'},
    'b':   {'a': 'a',   'a-1': 'a-1', 'b': 'b',   'b-1': 'D'},
    'b-1': {'a': 'a',   'a-1': 'a-1', 'b': 'D',   'b-1': 'b-1'},
    'D':   {'a': 'D',   'a-1': 'D',   'b': 'D',   'b-1': 'D'},
}
FINAL = {'S', 'a', 'a-1', 'b', 'b-1'}   # tous les etats sauf l'etat mort D

def is_geodesic(word):
    """Vrai si `word` est librement reduit (= geodesique). Decision en O(|word|), memoire constante."""
    s = 'S'
    for L in word:
        s = DELTA[s][L]
    return s in FINAL

# --- Verification PAR LEURS PROPRIETES (assertees, pas supposees) ---
# (1) un mot reduit est accepte
assert is_geodesic(word('aba')) is True
assert is_geodesic(word('b'))   is True
assert is_geodesic(word(''))    is True          # mot vide = geodesique de longueur 0
# (2) un mot avec annulation est rejete
assert is_geodesic(word('aa-1'))  is False
assert is_geodesic(word('ab-1b')) is False       # b-1 puis b s'annulent
assert is_geodesic(word('ba-1a')) is False
# (3) fermeture par produit : w geodesique, w·g l'est sauf si g annule la derniere lettre de w
for wc in ['a', 'ab', 'aba', 'b-1', 'ab-1']:
    w = word(wc)
    for g in ['a', 'a-1', 'b', 'b-1']:
        assert is_geodesic(w + [g]) == (g != INV[w[-1]]), (wc, g)

print("Accepteur Z^2 : 6 etats (S, a, a-1, b, b-1, D), 5 acceptants.")
print("Toutes les proprietes sont verifiees par les assertions ci-dessus.")


Accepteur Z^2 : 6 etats (S, a, a-1, b, b-1, D), 5 acceptants.
Toutes les proprietes sont verifiees par les assertions ci-dessus.


### Lecture du résultat

L'automate a **6 états** — bornés, indépendants de la longueur du mot. Il décide en une seule passe
gauche-droite, en **O(|mot|)** et en **mémoire constante** (un seul état courant). Les trois propriétés
demandées par #25674 sont vérifiées par les `assert` : un mot réduit est accepté, un mot qui annule est
rejeté, et l'ensemble des mots acceptés est *fermé par produit* — multiplier un mot géodésique par un
générateur le laisse géodésique, sauf quand ce générateur annule la dernière lettre.


## 3. Les cônes géodésiques — la régularité, et l'automate qu'elle donne

Le **cône géodésique** de `Z^2` est l'ensemble des mots géodésiques ; dire que son langage est *régulier*
c'est dire qu'un automate fini le reconnaît. L'automate ci-dessus **est** ce témoin : sa table de
transition est finie et effective, pas une affirmation.


In [3]:
# La table de transition FINIE de l'accepteur geodesique de Z^2 : la regularite, effective.
Lettres = ['a', 'a-1', 'b', 'b-1']
print("  \\  | " + " | ".join(L.rjust(4) for L in Lettres))
print("-" * 42)
for s in ['S', 'a', 'a-1', 'b', 'b-1', 'D']:
    row = "   ".join(DELTA[s][L].rjust(4) for L in Lettres)
    mark = "   <== acceptant" if s in FINAL else ("   (etat mort)" if s == 'D' else "")
    print(f"  {s:>3} | {row}{mark}")
print(f"\nNombre d'etats : {len(DELTA)} (borné, indépendant de la longueur des mots).")


  \  |    a |  a-1 |    b |  b-1
------------------------------------------
    S |    a    a-1      b    b-1   <== acceptant
    a |    a      D      b    b-1   <== acceptant
  a-1 |    D    a-1      b    b-1   <== acceptant
    b |    a    a-1      b      D   <== acceptant
  b-1 |    a    a-1      D    b-1   <== acceptant
    D |    D      D      D      D   (etat mort)

Nombre d'etats : 6 (borné, indépendant de la longueur des mots).


## 4. Le contraste qui compte — BFS (explorer) vs accepteur (décider)

Pour savoir si un mot de longueur `R` est géodésique, deux stratégies :

- **BFS** sur le graphe de Cayley de `Z^2` : on explore la boule de rayon `R`, qui compte **Θ(R²)**
  noeuds — on *explore* pour décider ;
- **accepteur** : on lit le mot en **O(|mot|) = O(R)**, *sans explorer*.

Le rapport *noeuds-BFS / états-accepteur* sur des sphères croissantes fait voir l'écart se creuser.


In [4]:
from collections import deque

def bfs_ball_z2(R):
    """Noeuds de la boule de rayon R dans le graphe de Cayley de Z^2 (comptés par BFS)."""
    seen = {(0, 0)}
    q = deque([(0, 0, 0)])
    while q:
        x, y, d = q.popleft()
        if d == R:
            continue
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            n = (x + dx, y + dy)
            if n not in seen:
                seen.add(n)
                q.append((n[0], n[1], d + 1))
    return len(seen)

# Un mot geodesique w_R = a^R de longueur R : BFS explore la boule, l'accepteur lit le mot.
print("   R  |  BFS noeuds  |  accepteur etats  |  rapport BFS/accepteur")
print("      |              |                   |" + " " * 8)
for R in [1, 2, 3, 5, 8, 12, 20, 30]:
    bfs_cost = bfs_ball_z2(R)          # ~ 2R^2 + 2R + 1  (quadratique)
    acc_cost = R + 1                   # |w_R| + 1 etats parcourus (linéaire)
    print(f"  {R:3d}  |  {bfs_cost:9d}  |  {acc_cost:15d}  |  {bfs_cost / acc_cost:8.1f} x")
print("\nLe BFS croit en ~2R^2 (quadratique), l'accepteur en R (linéaire) : le rapport diverge en ~2R.")


   R  |  BFS noeuds  |  accepteur etats  |  rapport BFS/accepteur
      |              |                   |        
    1  |          5  |                2  |       2.5 x
    2  |         13  |                3  |       4.3 x
    3  |         25  |                4  |       6.2 x
    5  |         61  |                6  |      10.2 x
    8  |        145  |                9  |      16.1 x
   12  |        313  |               13  |      24.1 x
   20  |        841  |               21  |      40.0 x
   30  |       1861  |               31  |      60.0 x

Le BFS croit en ~2R^2 (quadratique), l'accepteur en R (linéaire) : le rapport diverge en ~2R.


### Lecture du résultat

Le BFS visite `2R² + 2R + 1` noeuds pour couvrir la boule de rayon `R` (d'où le `~2R²`), tandis que
l'accepteur ne parcourt que `R + 1` états. Le rapport `BFS/accepteur` croît donc **linéairement avec R**
(colonne de droite) : à `R = 30` le BFS explore ~60 fois plus d'objets que l'automate n'en lit. C'est la
frontière « décider vs explorer » faite chiffre — la structure du groupe rend l'exploration *inutile*.


## 5. La croissance — polynôme (Z²) vs exponentielle (F₂), **comptée**

La signature d'un groupe automatique se lit dans la **croissance** de sa boule. On ne *cite* pas les
formules : on **compte** les sphères et les boules par BFS sur chaque graphe de Cayley, pour `k = 0..8`.


In [5]:
def sphere_z2(k):
    """Elements de Z^2 a distance EXACTEMENT k : |x| + |y| = k (comptés)."""
    n = 0
    for x in range(-k, k + 1):
        ry = k - abs(x)
        n += 1 if ry == 0 else 2
    return n

def ball_z2(k):
    return sum(sphere_z2(j) for j in range(k + 1))

def sphere_f2(k):
    """Mots librement reduits de longueur EXACTEMENT k dans F_2 (comptés par BFS sur l'arbre)."""
    level = [[]]                       # chaque mot = liste de tokens
    for _ in range(k):
        nxt = []
        for w in level:
            last = w[-1] if w else None
            for L in ['a', 'a-1', 'b', 'b-1']:
                if last is not None and L == INV[last]:
                    continue
                nxt.append(w + [L])
        level = nxt
    return len(level)

def ball_f2(k):
    return sum(sphere_f2(j) for j in range(k + 1))

print("  k  |  Z^2 sphere |  Z^2 boule  |   F2 sphere  |   F2 boule")
print("     |            |             |              |")
for k in range(0, 9):
    print(f"  {k:2d}  |  {sphere_z2(k):8d}  |  {ball_z2(k):8d}  |  {sphere_f2(k):10d}  |  {ball_f2(k):10d}")

# La signature : Z^2 quadratique (2e difference finie constante), F_2 exponentielle (ratio ~3).
assert ball_z2(10) == 2 * 10**2 + 2 * 10 + 1     # 221 : croissance polynomiale (degre 2)
assert ball_f2(5)  == 2 * 3**5 - 1               # 485 : croissance exponentielle (base 3)
print("\nZ^2 : boule = 2k^2 + 2k + 1  (polynome, degre 2).   F_2 : boule = 2·3^k - 1  (exponentielle, base 3).")


  k  |  Z^2 sphere |  Z^2 boule  |   F2 sphere  |   F2 boule
     |            |             |              |
   0  |         1  |         1  |           1  |           1
   1  |         4  |         5  |           4  |           5
   2  |         8  |        13  |          12  |          17
   3  |        12  |        25  |          36  |          53
   4  |        16  |        41  |         108  |         161
   5  |        20  |        61  |         324  |         485
   6  |        24  |        85  |         972  |        1457
   7  |        28  |       113  |        2916  |        4373
   8  |        32  |       145  |        8748  |       13121

Z^2 : boule = 2k^2 + 2k + 1  (polynome, degre 2).   F_2 : boule = 2·3^k - 1  (exponentielle, base 3).


### Lecture du résultat

La table fait la différence sans l'affirmer : la boule de `Z^2` suit `2k² + 2k + 1` (les *deuxièmes
différences finies* sont constantes — signature d'une croissance **polynomiale de degré 2**), alors que
celle de `F_2` suit `2·3^k − 1` (le *ratio* d'une boule à la précédente tend vers 3 — signature d'une
croissance **exponentielle**). C'est exactement le contraste que la théorie des groupes automatiques met
en scène : `Z^2` est automatique à croissance polynomiale, `F_2` est automatique à croissance
exponentielle — et les deux admettent un automate fini de mots géodésiques.


## Synthèse — où se lit la frontière « décider / explorer »

| Question | Sans structure (Search-02/03) | Avec l'automate du groupe (ce carnet) |
|---|---|---|
| « ce mot est-il géodésique ? » | BFS jusqu'au rayon \|mot\| — **Θ(\|mot\|²)** noeuds | lecture gauche-droite du DFA — **O(\|mot\|)**, mémoire constante |
| Coût en croissance | explore la boule (polynôme pour Z², exponentiel pour F₂) | indépendant de la boule — linéaire en le mot |

Quand le groupe *fournit* l'automate, la géométrie qui coûtait une exploration entière se réduit à un
test d'appartenance à un langage régulier. C'est le sens du « word processing » : le groupe fait le travail
de l'heuristique.


## Exercices


In [6]:
# Exercice 1 — L'accepteur géodésique de Z^3
def geodesic_accepter_z3():
    """Construire le DFA des mots geodesiques (librement reduits) de Z^3 : 3 generateurs, 6 lettres
    {a, a-1, b, b-1, c, c-1}. Combien d'etats faut-il ?
    Indice : comme pour Z^2, l'etat courant est la derniere lettre lue (ou le depart, ou l'etat mort)."""
    # TODO etudiant
    return None  # TODO etudiant


In [7]:
# Exercice 2 — Le ratio BFS / accepteur sur F_2
def ratio_bfs_accepteur_f2(R):
    """Pour un mot geodesique w de longueur R dans F_2, mesurer le rapport
    (noeuds explores par un BFS sur le graphe de Cayley de F_2) / (etats parcourus par l'accepteur).
    Deccrire comment ce ratio croit avec R, et le relier a la croissance exponentielle de F_2 (section 5)."""
    # TODO etudiant
    return None  # TODO etudiant


In [8]:
# Exercice 3 — Le probleme du mot COMPLET de Z^2 n'est pas regulier
def pourquoi_pas_regulier():
    """Le langage des mots qui VALENT l'identite dans Z^2 (somme des exposants = (0,0), cf. eval_z2)
    n'est PAS regulier, alors que le langage des mots GEODESIQUES l'est. Expliquer pourquoi :
    restreindre a l'alphabet {a, a-1} donne le langage des mots 'equilibres' (#a = #a-1), qui n'est pas
    regulier (lemme de la pompe). Pourquoi cette restriction ne contredit-elle pas la regularite du
    cône geodesique ?"""
    # TODO etudiant
    return None  # TODO etudiant


## À retenir

- Le langage des mots géodésiques de `Z^2` est **régulier** ; l'automate qui le reconnaît a **6 états**
  et se **construit** (sections 2–3) — il ne se suppose pas.
- Décider « géodésique ? » par cet automate coûte **O(|mot|)** ; le BFS explore **Θ(|mot|²)** — l'écart
  diverge (section 4).
- `Z^2` croît **polynomialement** (`2k²+2k+1`), `F_2` **exponentiellement** (`2·3^k−1`) — compté, pas cité
  (section 5).
- La structure algébrique du groupe, quand elle fournit l'automate, *remplace* l'exploration par une
  décision : c'est le fil qui relie ce carnet à Search-02/03.


## Bibliographie

- D. Epstein, J. W. Cannon, D. D. Holt, S. Levy, K. C. Paterson, W. P. Thurston, *Word Processing in
  Groups*, 1992 — le « word processing » et les cônes géodésiques.
- M. Gromov, *Groups that act on hyperbolic spaces*, Invent. Math. 1987 — structures automatiques.
- Carnets frères : [Search-02 — Recherche non informée](Search-02-Uninformed.ipynb) (BFS) ·
  [Search-03 — Recherche informée](Search-03-Informed.ipynb) (A*).
